In [1]:
!pip install tensorflow opencv-python numpy pillow scikit-learn openpyxl

In [2]:
import os, random, json, csv
import numpy as np
import cv2
from PIL import Image
from datetime import datetime
import openpyxl

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
STUDENTS_FOLDER = 'Faces'
IMG_SIZE        = (128, 128)
SEED            = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [4]:
def load_img(path):
    img = Image.open(path).convert('RGB').resize(IMG_SIZE)
    return np.array(img).astype(np.float32) / 255.0

X      = []
labels = []

for filename in os.listdir(STUDENTS_FOLDER):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        path = os.path.join(STUDENTS_FOLDER, filename)
        name = os.path.splitext(filename)[0]
        name = '_'.join(name.split('_')[:-1])
        X.append(load_img(path))
        labels.append(name)

X = np.array(X)
print('Dataset shape:', X.shape)

Dataset shape: (2377, 128, 128, 3)


In [5]:
le      = LabelEncoder()
y       = le.fit_transform(labels)
n_class = len(le.classes_)

print("Number of students:", n_class)
print("Classes:", le.classes_)

Number of students: 29
Classes: ['Alia Bhatt' 'Amitabh Bachchan' 'Andy Samberg' 'Anushka Sharma'
 'Billie Eilish' 'Brad Pitt' 'Camila Cabello' 'Charlize Theron'
 'Claire Holt' 'Courtney Cox' 'Dwayne Johnson' 'Elizabeth Olsen'
 'Ellen Degeneres' 'Henry Cavill' 'Hrithik Roshan' 'Hugh Jackman'
 'Jessica Alba' 'Kashyap' 'Lisa Kudrow' 'Margot Robbie' 'Marmik'
 'Natalie Portman' 'Priyanka Chopra' 'Robert Downey Jr' 'Roger Federer'
 'Tom Cruise' 'Vijay Deverakonda' 'Virat Kohli' 'Zac Efron']


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range     = 15,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    brightness_range   = [0.8, 1.2],
    horizontal_flip    = True,
    zoom_range         = 0.1
)

X_aug = []
y_aug = []

for img, label in zip(X, y):
    img_exp  = np.expand_dims(img, axis=0)
    aug_iter = datagen.flow(img_exp, batch_size=1)
    for _ in range(5):
        aug_img = next(aug_iter)[0]
        X_aug.append(aug_img)
        y_aug.append(label)  # ← نفس الـ label لكل صورة جديدة

X_aug = np.array(X_aug)
y_aug = np.array(y_aug)

X = np.concatenate([X, X_aug])
y = np.concatenate([y, y_aug])

print('X shape:', X.shape)
print('y shape:', y.shape)
print('يجب يكونوا نفس الرقم الأول ✅')

X shape: (14262, 128, 128, 3)
y shape: (14262,)
يجب يكونوا نفس الرقم الأول ✅


In [7]:
print('Unique students:', set(labels))

Unique students: {'Charlize Theron', 'Robert Downey Jr', 'Henry Cavill', 'Camila Cabello', 'Alia Bhatt', 'Kashyap', 'Brad Pitt', 'Virat Kohli', 'Ellen Degeneres', 'Anushka Sharma', 'Claire Holt', 'Roger Federer', 'Zac Efron', 'Lisa Kudrow', 'Hugh Jackman', 'Marmik', 'Andy Samberg', 'Margot Robbie', 'Natalie Portman', 'Amitabh Bachchan', 'Dwayne Johnson', 'Courtney Cox', 'Billie Eilish', 'Jessica Alba', 'Elizabeth Olsen', 'Priyanka Chopra', 'Vijay Deverakonda', 'Tom Cruise', 'Hrithik Roshan'}


In [8]:
# نطبع الطلاب الفريدين
print("Unique students:", set(labels))
print("n_class:", n_class)

Unique students: {'Charlize Theron', 'Robert Downey Jr', 'Henry Cavill', 'Camila Cabello', 'Alia Bhatt', 'Kashyap', 'Brad Pitt', 'Virat Kohli', 'Ellen Degeneres', 'Anushka Sharma', 'Claire Holt', 'Roger Federer', 'Zac Efron', 'Lisa Kudrow', 'Hugh Jackman', 'Marmik', 'Andy Samberg', 'Margot Robbie', 'Natalie Portman', 'Amitabh Bachchan', 'Dwayne Johnson', 'Courtney Cox', 'Billie Eilish', 'Jessica Alba', 'Elizabeth Olsen', 'Priyanka Chopra', 'Vijay Deverakonda', 'Tom Cruise', 'Hrithik Roshan'}
n_class: 29


In [9]:
print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (14262, 128, 128, 3)
y shape: (14262,)


In [10]:
le.classes_

array(['Alia Bhatt', 'Amitabh Bachchan', 'Andy Samberg', 'Anushka Sharma',
       'Billie Eilish', 'Brad Pitt', 'Camila Cabello', 'Charlize Theron',
       'Claire Holt', 'Courtney Cox', 'Dwayne Johnson', 'Elizabeth Olsen',
       'Ellen Degeneres', 'Henry Cavill', 'Hrithik Roshan',
       'Hugh Jackman', 'Jessica Alba', 'Kashyap', 'Lisa Kudrow',
       'Margot Robbie', 'Marmik', 'Natalie Portman', 'Priyanka Chopra',
       'Robert Downey Jr', 'Roger Federer', 'Tom Cruise',
       'Vijay Deverakonda', 'Virat Kohli', 'Zac Efron'], dtype='<U17')

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED
)

print('Train:', len(X_train))
print('Val  :', len(X_val))
print('Test :', len(X_test))

Train: 10303
Val  : 1819
Test : 2140


In [12]:
inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = Conv2D(32, 3, activation='relu', padding='same')(inp)
x = MaxPooling2D()(x)
x = Conv2D(64, 3, activation='relu', padding='same')(x)
x = MaxPooling2D()(x)
x = Conv2D(128, 3, activation='relu', padding='same')(x)

In [13]:
x = MaxPooling2D()(x)
x = Flatten()(x)
x = Dropout(0.3)(x)
x   = Dense(128, activation='relu')(x)
out = Dense(n_class, activation='softmax')(x)

In [14]:
X

array([[[[0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         ...,
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ]],

        [[0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         ...,
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ]],

        [[0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         ...,
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ]],

        ...,

        [[0.14509805, 0.11764706, 0.08627451],
         [0.15686275, 0.13333334, 0.10196079]

In [15]:
model = Model(inp, out)
model.compile(
    optimizer = 'adam',
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,291,421 (16.37 MB)

 Trainable params: 4,291,421 (16.37 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    X_train, y_train,
    epochs          = 50,
    batch_size      = 8,
    validation_data = (X_val, y_val)
)

Epoch 1/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 118s 90ms/step - accuracy: 0.0552 - loss: 3.3188 - val_accuracy: 0.0649 - val_loss: 3.2839
Epoch 2/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 159s 124ms/step - accuracy: 0.0839 - loss: 3.2057 - val_accuracy: 0.0973 - val_loss: 3.1916
Epoch 3/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 123s 95ms/step - accuracy: 0.1495 - loss: 2.9624 - val_accuracy: 0.1209 - val_loss: 3.2004
Epoch 4/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 116s 90ms/step - accuracy: 0.2312 - loss: 2.6698 - val_accuracy: 0.1391 - val_loss: 3.4108
Epoch 5/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 106s 82ms/step - accuracy: 0.2896 - loss: 2.4566 - val_accuracy: 0.1303 - val_loss: 3.7348
Epoch 6/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 104s 81ms/step - accuracy: 0.3265 - loss: 2.3362 - val_accuracy: 0.1380 - val_loss: 4.0854
Epoch 7/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 107s 83ms/step - accuracy: 0.3463 - loss: 2.2779 - val_accuracy: 0.1391 - val_loss: 4.0533
Epoch 8/50
1288/1288 ━━━━━━━━━━━━━━━━━━━━ 102s 79ms/step - accuracy:

KeyboardInterrupt: 

In [17]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
loss, acc 

(4.814090728759766, 0.15233644843101501)

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
print('\nReport:\n', classification_report(y_test, y_pred, target_names=le.classes_))

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step

Report:
                    precision    recall  f1-score   support

       Alia Bhatt       0.00      0.00      0.00         2
 Amitabh Bachchan       0.87      0.93      0.90        14
     Andy Samberg       1.00      0.70      0.82        10
   Anushka Sharma       0.50      0.40      0.44        15
    Billie Eilish       0.41      0.64      0.50        14
        Brad Pitt       0.94      0.68      0.79        22
   Camila Cabello       0.88      0.58      0.70        12
  Charlize Theron       0.82      0.70      0.76        20
      Claire Holt       0.57      0.50      0.53         8
     Courtney Cox       0.22      0.57      0.32         7
   Dwayne Johnson       1.00      0.40      0.57        10
  Elizabeth Olsen       0.67      0.57      0.62         7
  Ellen Degeneres       0.80      0.40      0.53        10
     Henry Cavill       0.56      0.59      0.57        17
   Hrithik Roshan       1.00      0.50      0.67        16
     

c:\Users\con027931\AppData\Local\miniconda3\envs\env_studying\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\con027931\AppData\Local\miniconda3\envs\env_studying\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\con027931\AppData\Local\miniconda3\envs\env_studying\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [ ]:
feature_extractor = Model(inputs=model.input, outputs=model.layers[-2].output)
feature_extractor

<Functional name=functional_2, built=True>

In [ ]:
import pandas as pd

df = pd.read_excel(r'D:/Dev/datascience-course/final_project2/Students_Images/Students.xlsx')

student_features = {}

for _, row in df.iterrows():
    name     = str(row['Name'])
    img_path = str(row['Image Path'])

    if not os.path.exists(img_path):
        print(f'Image not found: {img_path}')
        continue

    img = load_img(img_path)
    img = np.expand_dims(img, axis=0)

    features = feature_extractor.predict(img, verbose=0).flatten()

    if name in student_features:
        student_features[name] = (student_features[name] + features) / 2
    else:
        student_features[name] = features

class_names = list(student_features.keys())
class_names

['Malak', 'malak', 'ali']

In [ ]:
import json

model.save('face_model.keras')

embeddings = {name: feat.tolist() for name, feat in student_features.items()}
with open('student_features.json', 'w') as f:
    json.dump(embeddings, f)

In [ ]:
!pip install mediapipe --user

In [ ]:
import cv2
import numpy as np

def detect_drowsiness(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
    
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    if len(faces) == 0:
        return 'No Face'
    
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        eyes     = eye_cascade.detectMultiScale(roi_gray)
        
        if len(eyes) >= 2:
            return 'Active'
        else:
            return 'Sleepy'
    
    return 'No Face'



In [ ]:
import json

model.save("face_model.keras")

embeddings = {name: feat.tolist() for name, feat in student_features.items()}
with open("student_features.json", "w") as f:
    json.dump(embeddings, f)
